# CCTV Pedestrian & Crosswalk Detection — YOLOv5 Training

This notebook trains a YOLOv5 model to detect:
- **Class 0 — person**: individual pedestrians (one box per person)
- **Class 1 — crosswalk**: pedestrian crossing area (ONE box per crossing)

**Dataset**:
- COCO 2017 (filtered to `person` only) for general pedestrian appearance
- Custom CCTV dataset (≥ 800 images) with `person` + `crosswalk` labels

**Key settings used in this notebook:**
- `--img 640` — standard resolution for CCTV footage
- `--batch 16` — fits T4 GPU at 640 px with `yolov5s`
- `--weights yolov5s.pt` — fast and accurate baseline
- `--cache ram` — caches images in RAM for faster per-epoch training
- `--workers 8` — 8 parallel data-loader workers

## Step 1 — Check GPU

In [ ]:
!nvidia-smi

## Step 2 — Clone YOLOv5 and install requirements

In [ ]:
import os

if not os.path.exists('/content/yolov5'):
    !git clone https://github.com/ultralytics/yolov5 /content/yolov5
else:
    print('yolov5 already cloned')

%cd /content/yolov5
!pip install -r requirements.txt -q

## Step 3 — Clone this repository

In [ ]:
if not os.path.exists('/content/Activity_yolov5'):
    !git clone https://github.com/Carfok/Activity_yolov5 /content/Activity_yolov5
else:
    print('Activity_yolov5 already cloned')

## Step 4 — Download COCO 2017 and convert to YOLO (person only)

This step downloads COCO 2017 validation images (~1 GB) and annotations,
then converts person annotations to YOLO format using the provided script.

> **Note:** Downloading the full training set (~19 GB) takes a long time.
> For a quick experiment you can use only the val split or a subset.

In [ ]:
import os

COCO_DIR = '/content/coco'
os.makedirs(f'{COCO_DIR}/images', exist_ok=True)
os.makedirs(f'{COCO_DIR}/annotations', exist_ok=True)

# Download and unzip COCO 2017 val images (~1 GB)
if not os.path.exists(f'{COCO_DIR}/images/val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O /tmp/val2017.zip
    !unzip -q /tmp/val2017.zip -d {COCO_DIR}/images/
    print('val2017 images ready')

# (Optional) Download train2017 images — large (~19 GB), comment out if not needed
# if not os.path.exists(f'{COCO_DIR}/images/train2017'):
#     !wget -q http://images.cocodataset.org/zips/train2017.zip -O /tmp/train2017.zip
#     !unzip -q /tmp/train2017.zip -d {COCO_DIR}/images/

# Download annotations
if not os.path.exists(f'{COCO_DIR}/annotations/instances_val2017.json'):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /tmp/annotations.zip
    !unzip -q /tmp/annotations.zip -d {COCO_DIR}/
    print('annotations ready')

In [ ]:
# Convert COCO val annotations to YOLO format (person only)
!python /content/Activity_yolov5/scripts/convert_coco_person_to_yolo.py \
    --coco_dir /content/coco \
    --output_dir /content/datasets/cctv_crosswalk \
    --splits val \
    --copy_images \
    --no_empty

# If you also downloaded train2017, add --splits train val above

## Step 5 — Upload custom CCTV dataset

Upload your CCTV images and YOLO labels (`.txt`) to the matching directories:

```
/content/datasets/cctv_crosswalk/
├── images/{train,val,test}/   ← your CCTV images
└── labels/{train,val,test}/   ← your YOLO label .txt files
```

Label format per line: `<class_id> <x_center> <y_center> <width> <height>` (normalised)
- class `0` = person
- class `1` = crosswalk (one box per crossing)

In [ ]:
# Option A: upload a zip from your local machine
from google.colab import files
uploaded = files.upload()   # select your cctv_crosswalk.zip

import zipfile, os
for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/datasets/cctv_crosswalk')
        print(f'Extracted {fname}')

# Option B: mount Google Drive and copy from there
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/cctv_crosswalk /content/datasets/

## Step 6 — Create the dataset YAML config

In [ ]:
import yaml

cfg = {
    'path':  '/content/datasets/cctv_crosswalk',
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    2,
    'names': ['person', 'crosswalk'],
}

yaml_path = '/content/Activity_yolov5/data/cctv_crosswalk.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f'Wrote {yaml_path}')
!cat {yaml_path}

## Step 7 — Verify dataset structure

In [ ]:
from pathlib import Path
from collections import Counter

CLASS_NAMES = {0: 'person', 1: 'crosswalk'}
dataset_root = Path('/content/datasets/cctv_crosswalk')

for split in ['train', 'val', 'test']:
    img_dir = dataset_root / 'images' / split
    lbl_dir = dataset_root / 'labels' / split
    if not img_dir.exists():
        print(f'  [{split}] directory not found — skipping')
        continue

    img_stems = {f.stem for f in img_dir.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png'}}
    lbl_stems = {f.stem for f in lbl_dir.glob('*.txt')} if lbl_dir.exists() else set()
    matches   = img_stems & lbl_stems

    dist = Counter()
    for lf in lbl_dir.glob('*.txt') if lbl_dir.exists() else []:
        for line in lf.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                dist[int(parts[0])] += 1

    print(f'--- {split} ---')
    print(f'  Images : {len(img_stems)}')
    print(f'  Labels : {len(lbl_stems)}')
    print(f'  Matched: {len(matches)}')
    for cid in sorted(dist):
        print(f'  Class {cid} ({CLASS_NAMES.get(cid, "?")}) : {dist[cid]} instances')

## Step 8 — Train YOLOv5

Key flags:
| Flag | Value | Reason |
|------|-------|--------|
| `--img` | `640` | Standard resolution for CCTV footage |
| `--batch` | `16` | Fits T4 GPU at 640 px with `yolov5s` |
| `--epochs` | `100` | Good starting point; increase for larger datasets |
| `--weights` | `yolov5s.pt` | Small but fast baseline |
| `--cache` | `ram` | Cache images in RAM for faster per-epoch training |
| `--workers` | `8` | 8 parallel data-loader workers |


In [ ]:
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 100 \
    --data /content/Activity_yolov5/data/cctv_crosswalk.yaml \
    --weights yolov5s.pt \
    --project /content/runs/train \
    --name cctv_crosswalk \
    --cache ram \
    --workers 8 \
    --exist-ok

## Step 9 — Review training results

In [ ]:
from IPython.display import Image as IPImage, display
import glob

results_dir = '/content/runs/train/cctv_crosswalk'

for img_path in sorted(glob.glob(f'{results_dir}/*.png')):
    print(img_path)
    display(IPImage(img_path))

## Step 10 — Validate best model

In [ ]:
%cd /content/yolov5

!python val.py \
    --img 640 \
    --data /content/Activity_yolov5/data/cctv_crosswalk.yaml \
    --weights /content/runs/train/cctv_crosswalk/weights/best.pt \
    --task val \
    --verbose

## Step 11 — Run inference on a sample image or video

In [ ]:
%cd /content/yolov5

# Replace the --source value with your image, video, or webcam index (0)
!python detect.py \
    --weights /content/runs/train/cctv_crosswalk/weights/best.pt \
    --source  /content/datasets/cctv_crosswalk/images/test \
    --data    /content/Activity_yolov5/data/cctv_crosswalk.yaml \
    --img     640 \
    --save-txt \
    --project /content/runs/detect \
    --name    cctv_crosswalk

## Step 12 — Download trained weights

In [ ]:
from google.colab import files

files.download('/content/runs/train/cctv_crosswalk/weights/best.pt')